# 19. Re-pick the ELECTRA blend weight on training rows

Owner: Jalil-g. Issue #37. Depends on completed #18.

## Goal

#18 picked the ELECTRA blend weight (.25) on frozen 2000 validation rows, which breaks the training-only tuning rule. Here we redo the same pick on the inner 1600 training rows and check if it stays .25.

## Rule (fixed before running)

**Rows.** Inner 6400/1600 inside original 8000, the same rows as #12, #14, #16, #17. Classical uses the #17 row order, ELECTRA the #16 row order. Both must be the same ID sets.

**Components.** Classical = #17 pick `hist50_old50` (augmented style .5, old soft text .5), exact #17/#18 code and settings. ELECTRA = #16 code, 3 epochs (the #16 internal pick), seed 42. Both fit on inner 6400 only.

**Scores.** Each component score is standardized on eligible unlabeled rows (mean, population std), using the 6400-fit models. Classical views use the #17/#18 views (wrap80 = paragraph wrap). ELECTRA views use the #16 views (wrap80 = line wrap), same as #18.

**Grid and pick.** Exactly #18: score = (1 - w) * classical + w * ELECTRA for w in 0, .1, .25, .5, .75, 1. Objective = mean AUROC over original, remove_blank, collapse, wrap80. Guardrail: original >= w=0 original - .002. Exact ties go to smaller w. Pick is saved before frozen 2000 is read.

**After the pick.** Frozen 2000 confirms the pick with the saved 8000-fit scores from #16 and #17. It was reused before, so it is optimistic, but it no longer picks anything. Then the saved #18 10000-fit models score test: w = .25 must reproduce the submitted file, and if the new pick differs, a new file is written. No upload.

## Setup

Helpers come from notebooks 10 and 13 in helpers-only mode, so nothing there fits or writes. Arm code is copied from #18 and ELECTRA code from #16/#18. Runs in `envs/electra` because ELECTRA needs torch.

In [ ]:
import os, json, time, math, textwrap, warnings, platform, subprocess
from contextlib import contextmanager
import cloudpickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.special import expit, logit
from IPython import get_ipython
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from transformers.utils import logging as hf_logging
os.environ["EXPANDED_TEXT_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./10_expanded_text.ipynb")
os.environ["RICH_STYLE_HELPERS_ONLY"] = "1"
get_ipython().run_line_magic("run", "./13_rich_style.ipynb")
hf_logging.set_verbosity_error()

NAME = "19_internal_neural_pick"
COMMAND = "uv run --project envs/electra --locked jupyter nbconvert --execute --to notebook --inplace --ExecutePreprocessor.timeout=7200 notebooks/19_internal_neural_pick.ipynb"
# Classical pick from #17, settings as in #18.
STYLE_HIST_SETTINGS = {"max_iter": 250, "learning_rate": .06, "max_leaf_nodes": 15, "min_samples_leaf": 30,
                       "l2_regularization": 5., "early_stopping": False, "random_state": 42}
COPY_WEIGHT, STYLE_COPIES = .25, ["collapse"]
CLASSICAL = {"selected": "hist50_old50", "weights": [.5, .5, 0.]}
OLD_SOFT = {"word": {"view": "word_char", "learner": "lr", "C": 16.}, "specialist": {"view": "casechar", "learner": "svm", "C": 1.}}
# Neural grid and rule, same as #18.
NEURAL_WEIGHTS, NEURAL_SLACK, OLD_PICK = [0, .1, .25, .5, .75, 1], .002, .25
VIEWS = ["original", "remove_blank", "collapse", "wrap80"]
ELECTRA_VIEW = {"original": "original", "remove_blank": "remove_blank_lines", "collapse": "collapse_whitespace", "wrap80": "wrap80"}
# ELECTRA settings from #16. 3 epochs was its internal pick.
MAX_TOKENS, BATCH, EVAL_BATCH, LR, WEIGHT_DECAY, WARMUP_FRACTION, SCHEDULE_EPOCHS, EPOCHS = 256, 16, 64, 3e-5, .01, .1, 3, 3
DEVICE = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
CKPT = ROOT / "artifacts" / "16_local_electra" / "checkpoint"
CHECKPOINT_FILES = ["config.json", "pytorch_model.bin", "tokenizer.json", "tokenizer_config.json", "vocab.txt"]
INPUT_RUNS = {"16": "20261110T222108811422Z", "17": "20261110T231659204936Z", "18": "20261111T002651559290Z"}
CONFIG = {
    "why": "#18 picked the ELECTRA weight on frozen 2000; this repeats the pick on inner training rows",
    "rows": {"fit": 6400, "pick": 1600, "inside": "original 8000", "classical_order": "#17 train_test_split recipe",
             "electra_order": "#16 split_ids recipe", "same_sets": True},
    "classical": CLASSICAL, "style": {"histgb": STYLE_HIST_SETTINGS, "copies": STYLE_COPIES, "copy_weight": COPY_WEIGHT},
    "old_soft": {**OLD_SOFT, "flag": "literal two consecutive newlines; active only when zero human fit rows have it", "offset": 2.},
    "electra": {"epochs": EPOCHS, "max_tokens": MAX_TOKENS, "batch": BATCH, "eval_batch": EVAL_BATCH, "lr": LR, "weight_decay": WEIGHT_DECAY,
                "warmup_fraction": WARMUP_FRACTION, "schedule_epochs": SCHEDULE_EPOCHS, "device": DEVICE,
                "nondeterminism": "MPS kernels are not bitwise deterministic"},
    "neural_blend": {"weights": NEURAL_WEIGHTS, "score": "(1-w)*std(classical) + w*std(ELECTRA), std on eligible unlabeled with 6400-fit models",
                     "objective": "mean AUROC over original, remove_blank, collapse, wrap80", "guardrail": "original >= w0 original - .002",
                     "ties": "smaller w", "rows": "inner 1600 (training rows)", "old_pick_on_frozen2000": OLD_PICK},
    "confirmation": "frozen 2000 with saved 8000-fit scores from #16 and #17, after the pick is saved",
    "test_rescore": "saved #18 10000-fit models; w=.25 must reproduce the submitted file; new file only if the pick changes",
    "upload": "never automatic; Jalil uploads by hand",
}
START = time.perf_counter()
print("Device:", DEVICE, "| torch", torch.__version__)

## Checks

Notebook 02 checks data bytes and the frozen split. Both inner splits must match their source records and each other. Dependency records, settings, the checkpoint, and every private input must match their recorded hashes.

In [ ]:
parts = load_partitions()
train, validation, reference = parts["train"], parts["validation"], parts["unlabeled"]
test, sample = read_raw("test.csv"), read_raw("sample_submission.csv")
validate_ids(test); validate_ids(sample)
require(len(test) == 3000 and set(test.id) == set(sample.id), "Sample and test IDs differ")

def latest_completed(folder):
    rows = [(p, json.loads(p.read_text())) for p in sorted((ROOT / "results" / folder).glob("2*Z.json"))]
    rows = [(p, r) for p, r in rows if r.get("status") == "completed"]
    require(bool(rows), f"Completed {folder} record is required")
    return rows[-1]

deps = {k: latest_completed(f) for k, f in [("16", "16_local_electra"), ("17", "17_final_blend"), ("18", "18_final_checks")]}
dep16, dep17, dep18 = (deps[k][1] for k in ["16", "17", "18"])
require(dep17["metrics"]["selected"]["candidate"] == CLASSICAL["selected"] and dep17["metrics"]["selected"]["weights"] == CLASSICAL["weights"], "#17 pick changed")
require(dep17["config"]["style"]["histgb"] == STYLE_HIST_SETTINGS and dep17["config"]["style"]["copies"] == STYLE_COPIES, "#17 style settings changed")
require(dep16["metrics"]["selected_epochs"] == EPOCHS and dep16["config"]["blend"]["neural_weights"] == NEURAL_WEIGHTS, "#16 settings changed")
t16 = dep16["config"]["train"]
require([t16[k] for k in ["max_tokens", "batch", "eval_batch", "lr", "weight_decay", "warmup_fraction", "schedule_epochs"]]
        == [MAX_TOKENS, BATCH, EVAL_BATCH, LR, WEIGHT_DECAY, WARMUP_FRACTION, SCHEDULE_EPOCHS], "#16 train settings changed")
require(dep18["metrics"]["neural_selected"]["neural_weight"] == OLD_PICK, "#18 pick is not .25")
require(dep18["config"]["neural_blend"]["weights"] == NEURAL_WEIGHTS, "#18 grid changed")
require(dep16["config"]["checkpoint"]["files_sha256"] == {n: file_hash(CKPT / n) for n in CHECKPOINT_FILES}, "Checkpoint files changed")

# Classical inner rows in #17 order.
a, b = train_test_split(np.arange(len(train)), test_size=.2, stratify=train.label, random_state=42)
a, b = np.sort(a), np.sort(b)
c_fit, c_eval = train.iloc[a].reset_index(drop=True), train.iloc[b].reset_index(drop=True)
require({"fit": value_hash(c_fit.id.tolist()), "score": value_hash(c_eval.id.tolist())} == dep17["config"]["internal_id_sha256"], "Classical inner split differs from #17")
# ELECTRA inner rows in #16 order.
_, components, _ = checked_audit(ROOT / parts["split"]["audit_path"])
train_ids = set(train.id)
inner_components = [[r for r in comp if r["file"] == "train.csv" and r["id"] in train_ids] for comp in components]
inner_components = [comp for comp in inner_components if len(comp) > 1]
fit_ids, eval_ids = split_ids(train[["id", "label"]], inner_components, validation_per_class=800, seed=SEED)
indexed = train.set_index("id", drop=False)
e_fit, e_eval = indexed.loc[fit_ids].reset_index(drop=True), indexed.loc[eval_ids].reset_index(drop=True)
require({"fit_ids_sha256": value_hash(fit_ids), "eval_ids_sha256": value_hash(eval_ids)}
        == {k: dep16["config"]["inner_split"][k] for k in ["fit_ids_sha256", "eval_ids_sha256"]}, "ELECTRA inner split differs from #16")
require(set(c_fit.id) == set(e_fit.id) and set(c_eval.id) == set(e_eval.id), "Classical and ELECTRA inner rows differ")
require(c_eval.label.value_counts().to_dict() == {0: 800, 1: 800} and not set(c_eval.id) & set(validation.id), "Bad inner eval rows")

# Saved 8000-fit frozen 2000 scores (confirmation only) and saved #18 10000-fit models (test rescore only).
inputs = {"17_unlabeled": f"artifacts/17_final_blend/{INPUT_RUNS['17']}/components_unlabeled.csv",
          **{f"17_validation_{v}": f"artifacts/17_final_blend/{INPUT_RUNS['17']}/components_validation_{v}.csv" for v in VIEWS},
          "16_unlabeled": f"artifacts/16_local_electra/{INPUT_RUNS['16']}/eligible_unlabeled_margins.csv",
          "16_validation": f"artifacts/16_local_electra/{INPUT_RUNS['16']}/frozen2000_margins.csv"}
for key, rel in inputs.items():
    record = dep17 if key.startswith("17") else dep16
    require(record["artifacts"].get(rel) == file_hash(ROOT / rel), f"Private input {rel} missing or changed")
full_dir = f"artifacts/18_final_checks/{INPUT_RUNS['18']}"
full_files = [f"{full_dir}/classical_10000.pkl", f"{full_dir}/electra_10000/config.json", f"{full_dir}/electra_10000/model.safetensors",
              f"submissions/18_final_checks/{INPUT_RUNS['18']}/submission.csv"]
for rel in full_files:
    require(dep18["artifacts"].get(rel) == file_hash(ROOT / rel), f"#18 file {rel} missing or changed")

CONFIG["dependencies"] = {k: {"record": str(p.relative_to(ROOT)), "record_sha256": file_hash(p)} for k, (p, _) in deps.items()}
CONFIG["private_inputs_sha256"] = {rel: file_hash(ROOT / rel) for rel in list(inputs.values()) + full_files}
CONFIG["inner_id_sha256"] = {"classical": dep17["config"]["internal_id_sha256"], "electra": {"fit": value_hash(fit_ids), "eval": value_hash(eval_ids)}}
CONFIG["reference"] = {"rows": len(reference), "ordered_id_sha256": value_hash(reference.id.tolist())}
CONFIG["checkpoint_sha256"] = dep16["config"]["checkpoint"]["files_sha256"]
CONFIG["helper_source_sha256"] = {n: value_hash(["".join(c["source"]) for c in json.loads((ROOT / "notebooks" / n).read_text())["cells"]])
                                  for n in ["10_expanded_text.ipynb", "13_rich_style.ipynb"]}
CONFIG["notebook_code_sha256"] = value_hash(["".join(c["source"]) for c in json.loads((ROOT / f"notebooks/{NAME}.ipynb").read_text())["cells"] if c["cell_type"] == "code"])
CONFIG["neural_lock"] = {"path": "envs/electra/uv.lock", "sha256": file_hash(ROOT / "envs/electra/uv.lock")}
CONFIG["neural_versions"] = {n: importlib.metadata.version(n) for n in ["torch", "transformers", "tokenizers", "safetensors", "cloudpickle", "threadpoolctl"]}
chip = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip() if platform.system() == "Darwin" else platform.processor()
CONFIG["hardware"] = {"device": DEVICE, "chip": chip, "platform": platform.platform(), "torch_threads": torch.get_num_threads()}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
private = ROOT / "artifacts" / NAME / RUN_ID
private.mkdir(parents=True, exist_ok=False)
print(f"Inner fit {len(c_fit)}, inner pick {len(c_eval)}, eligible unlabeled {len(reference)}, test {len(test)}")
print("Splits, dependencies, settings, checkpoint, and private inputs match.")

## Method

Classical arms are the #18 code. ELECTRA training and scoring are the #16/#18 code. `neural_grid` is the #18 function unchanged. Every standardization uses eligible unlabeled rows only.

In [ ]:
def literal_blank_flags(texts):
    return np.asarray(["\n\n" in str(t) for t in texts], dtype=bool)

def wrap_view(text, width):
    chunks = [c for c in PARAGRAPH_SPLIT.split(str(text)) if c.strip()]
    return "\n\n".join(textwrap.fill(c, width=width, break_long_words=False, break_on_hyphens=False) for c in chunks)

def stress_texts(texts, variant):
    # Classical views (#17/#18).
    if variant == "original": return [str(t) for t in texts]
    if variant == "remove_blank": return ["\n".join(line for line in str(t).split("\n") if line.strip()) for t in texts]
    if variant == "collapse": return [" ".join(str(t).split()) for t in texts]
    if variant == "wrap80": return [wrap_view(t, 80) for t in texts]
    raise ValueError("Unknown stress view")

# ELECTRA views (#16).
ELECTRA_STRESS = {"original": str,
                  "remove_blank_lines": lambda t: "\n".join(line for line in str(t).split("\n") if line.strip()),
                  "collapse_whitespace": lambda t: " ".join(str(t).split()),
                  "wrap80": lambda t: "\n".join(textwrap.fill(line, 80) if line.strip() else line for line in str(t).split("\n"))}

def ref_stats(values):
    values = np.asarray(values, dtype=float)
    stats = {"mean": float(values.mean()), "std": float(values.std())}
    require(stats["std"] > 1e-12, "Degenerate reference")
    return stats

def standardize(values, stats):
    return (np.asarray(values, dtype=float) - stats["mean"]) / stats["std"]

class StyleArm:
    """#13 HistGB on rich-style features with #14 plus_collapse copies (#17/#18 code)."""
    def fit(self, texts, labels, reference_texts):
        texts, labels = list(texts), np.asarray(labels)
        views = [texts] + [stress_texts(texts, v) for v in STYLE_COPIES]
        X, y = np.vstack([rich_style_features(v) for v in views]), np.concatenate([labels] * len(views))
        weights = np.concatenate([np.ones(len(labels))] + [np.full(len(labels), COPY_WEIGHT)] * len(STYLE_COPIES))
        self.model_ = HistGradientBoostingClassifier(**STYLE_HIST_SETTINGS).fit(X, y, sample_weight=weights)
        self.statistics_ = ref_stats(self.raw_margin(reference_texts))
        return self
    def raw_margin(self, texts): return logit(np.clip(class1_probabilities(self.model_, rich_style_features(list(texts))), 1e-7, 1 - 1e-7))
    def decision_function(self, texts): return standardize(self.raw_margin(texts), self.statistics_)

class SoftTextArm:
    """#11/#15 old soft text: word+char LR, no-blank char SVM, literal blank flag (#17/#18 code)."""
    def fit(self, texts, labels, reference_texts):
        texts = pd.Series(list(texts)); labels = np.asarray(labels); keep = ~literal_blank_flags(texts)
        self.rule_active_ = not bool(literal_blank_flags(texts[labels == 0]).any())
        require(set(labels[keep]) == {0, 1}, "No-blank specialist needs both classes")
        self.word_ = make_expanded_model(OLD_SOFT["word"]).fit(texts, labels)
        self.specialist_ = make_expanded_model(OLD_SOFT["specialist"]).fit(texts[keep], labels[keep])
        reference_texts = pd.Series(list(reference_texts))
        self.word_stats_ = margin_reference(self.word_, reference_texts)
        self.specialist_stats_ = margin_reference(self.specialist_, reference_texts[~literal_blank_flags(reference_texts)])
        self.outer_ = ref_stats(self.raw_margin(reference_texts))
        self.fit_rows_, self.specialist_rows_ = len(texts), int(keep.sum())
        return self
    def raw_margin(self, texts):
        score = .5 * (standardized_margin(self.word_, texts, self.word_stats_) + standardized_margin(self.specialist_, texts, self.specialist_stats_))
        return score + 2. * literal_blank_flags(texts) * self.rule_active_
    def decision_function(self, texts): return standardize(self.raw_margin(texts), self.outer_)

def fit_classical(frame, reference_frame):
    seconds = {}
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        t = time.perf_counter(); style = StyleArm().fit(frame.text, frame.label, reference_frame.text); seconds["style"] = time.perf_counter() - t
        t = time.perf_counter(); old = SoftTextArm().fit(frame.text, frame.label, reference_frame.text); seconds["old_soft"] = time.perf_counter() - t
    return style, old, seconds

def classical_score(style, old, texts):
    with threadpool_limits(limits=1):
        return CLASSICAL["weights"][0] * style.decision_function(texts) + CLASSICAL["weights"][1] * old.decision_function(texts)

tokenizer = AutoTokenizer.from_pretrained(CKPT, local_files_only=True)
def encode(texts): return tokenizer(list(map(str, texts)), truncation=True, max_length=MAX_TOKENS)["input_ids"]
def to_batch(input_ids):
    batch = tokenizer.pad({"input_ids": input_ids}, return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in batch.items()}

@torch.no_grad()
def electra_margins(model, texts):
    model.eval(); ids, out = encode(texts), []
    for i in range(0, len(ids), EVAL_BATCH):
        logits = model(**to_batch(ids[i:i + EVAL_BATCH])).logits.float()
        out.append((logits[:, 1] - logits[:, 0]).cpu().numpy())
    values = np.concatenate(out).astype(float)
    require(values.shape == (len(texts),) and np.isfinite(values).all(), "Bad margins")
    return values

def train_electra(fit, epochs):
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(CKPT, num_labels=2, local_files_only=True).to(DEVICE)
    ids, labels = encode(fit.text), torch.tensor(fit.label.to_numpy(), dtype=torch.long)
    steps_per_epoch = math.ceil(len(ids) / BATCH); total_steps = steps_per_epoch * SCHEDULE_EPOCHS
    warmup_steps = round(WARMUP_FRACTION * total_steps)
    decay = [p for n, p in model.named_parameters() if not (n.endswith("bias") or "LayerNorm" in n)]
    no_decay = [p for n, p in model.named_parameters() if n.endswith("bias") or "LayerNorm" in n]
    optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.}], lr=LR)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    generator, history = torch.Generator().manual_seed(SEED), []
    for epoch in range(1, epochs + 1):
        model.train(); order, losses, start = torch.randperm(len(ids), generator=generator).tolist(), [], time.perf_counter()
        for i in range(0, len(order), BATCH):
            idx = order[i:i + BATCH]
            loss = model(**to_batch([ids[j] for j in idx]), labels=labels[idx].to(DEVICE)).loss
            loss.backward(); optimizer.step(); scheduler.step(); optimizer.zero_grad(); losses.append(loss.item())
        if DEVICE == "mps": torch.mps.synchronize()
        history.append({"epoch": epoch, "steps_done": epoch * steps_per_epoch, "total_schedule_steps": total_steps, "warmup_steps": warmup_steps,
                        "mean_train_loss": float(np.mean(losses)), "train_seconds": time.perf_counter() - start})
        print({k: round(v, 6) if isinstance(v, float) else v for k, v in history[-1].items()})
    return model, history

def neural_grid(labels, classical, neural):
    """#18 rule. classical, neural: dict view -> standardized scores on the same rows."""
    rows = []
    for w in NEURAL_WEIGHTS:
        row = {"neural_weight": w, **{v: float(roc_auc_score(labels, (1 - w) * classical[v] + w * neural[v])) for v in VIEWS}}
        row["mean_auroc"] = float(np.mean([row[v] for v in VIEWS])); rows.append(row)
    floor = rows[0]["original"] - NEURAL_SLACK
    for row in rows: row["eligible"] = row["original"] >= floor
    best = max(r["mean_auroc"] for r in rows if r["eligible"])
    return rows, next(r for r in rows if r["eligible"] and r["mean_auroc"] == best)

def check_submission(path, sample_ids, expected):
    """#18 checker. expected: dict id -> P(AI) recomputed separately."""
    frame = pd.read_csv(path, dtype={"id": "string"})
    require(list(frame.columns) == ["id", "label"], "Columns must be exactly id,label")
    require(frame.id.notna().all() and frame.id.is_unique, "Missing or duplicate IDs")
    require(set(frame.id) == set(sample_ids) and len(frame) == len(sample_ids), "IDs must match sample exactly")
    require(np.isfinite(frame.label).all() and frame.label.between(0, 1).all(), "Labels must be finite in [0, 1]")
    diff = np.abs(frame.label.to_numpy() - frame.id.map(expected).to_numpy(dtype=float))
    require(float(diff.max()) <= 1e-12, "Values do not match recomputed predictions by ID")
    return {"rows": len(frame), "max_abs_diff_vs_recompute": float(diff.max())}

@contextmanager
def guarded(stage):
    try:
        yield
    except Exception as exc:
        record_run(NAME, COMMAND, CONFIG, SEED, 0, 0, {"validation_auroc": None, "failed_stage": stage},
                   time.perf_counter() - START, artifacts=artifacts, status="failed", error_type=type(exc).__name__)
        raise RuntimeError(f"Stage {stage} failed. See the safe failed record.") from None

# Synthetic checks of the #18 rule: tie order, robust win, guardrail.
y = np.array([0, 0, 1, 1]); flat = {v: np.array([0., 1., 2., 3.]) for v in VIEWS}
require(neural_grid(y, flat, flat)[1]["neural_weight"] == 0, "Exact tie must go to w=0")
weak = {**flat, "collapse": np.array([1., 0., .5, 2.])}
require(neural_grid(y, weak, flat)[1]["neural_weight"] > 0, "Robust neural score should win when classical breaks")
rows, pick = neural_grid(y, flat, {v: np.array([3., 2., 1., 0.]) for v in VIEWS})
require(pick["neural_weight"] == 0 and not rows[-1]["eligible"], "Guardrail must block a bad neural score")
print("Rule fixtures passed.")

## Run

Step 1: classical fit on inner 6400, check it reproduces #17 internal numbers. Step 2: ELECTRA fit on inner 6400. Step 3: grid and pick on inner 1600, saved before frozen 2000. Step 4: frozen 2000 confirmation and test rescore with the saved #18 models.

In [ ]:
plan_path = ROOT / "results" / NAME / f"{RUN_ID}-plan.json"
save_new_json(plan_path, {"config": CONFIG, "config_sha256": value_hash(CONFIG), "saved_before_fit": True, "split_sha256": parts["split"]["split_sha256"]})
artifacts, timings = [plan_path], {}

with guarded("classical_inner"):
    t = time.perf_counter()
    style6, old6, classical_fit_seconds = fit_classical(c_fit, reference)
    c_stats = ref_stats(classical_score(style6, old6, reference.text))
    classical_inner = {v: standardize(classical_score(style6, old6, stress_texts(c_eval.text, v)), c_stats) for v in VIEWS}
    timings["classical_inner_seconds"] = time.perf_counter() - t
    # Reproduction: same code and rows as #17 internal, so AUROC must match its saved grid.
    saved17 = next(r for r in dep17["metrics"]["internal_grid"] if r["candidate"] == CLASSICAL["selected"])
    classical_repro = {v: abs(float(roc_auc_score(c_eval.label, classical_inner[v])) - saved17[v]) for v in VIEWS if v in saved17}
    require(max(classical_repro.values()) <= 1e-9, f"Classical inner differs from #17: {classical_repro}")
print("Classical inner vs #17 internal grid, max AUROC diff:", max(classical_repro.values()))

In [ ]:
with guarded("electra_inner"):
    t = time.perf_counter()
    electra6, electra_history = train_electra(e_fit, EPOCHS)
    timings["electra_train_seconds"] = time.perf_counter() - t
    t = time.perf_counter()
    e_stats = ref_stats(electra_margins(electra6, reference.text))
    pos = pd.Series(np.arange(len(e_eval)), index=e_eval.id).loc[c_eval.id].to_numpy()  # ELECTRA rows into classical order
    electra_inner = {v: standardize(electra_margins(electra6, [ELECTRA_STRESS[ELECTRA_VIEW[v]](x) for x in e_eval.text]), e_stats)[pos] for v in VIEWS}
    timings["electra_score_seconds"] = time.perf_counter() - t
    electra_inner_auroc = {v: float(roc_auc_score(c_eval.label, electra_inner[v])) for v in VIEWS}
    saved16 = dep16["metrics"]["inner_history"][EPOCHS - 1]["auroc_original"]
    del electra6
print("ELECTRA inner AUROC:", {k: round(v, 6) for k, v in electra_inner_auroc.items()}, f"| #16 epoch 3 original {saved16:.6f}")

In [ ]:
with guarded("inner_pick"):
    inner_rows, inner_pick = neural_grid(c_eval.label.to_numpy(), classical_inner, electra_inner)
    W = float(inner_pick["neural_weight"])
    selection_path = ROOT / "results" / NAME / f"{RUN_ID}-selection.json"
    save_new_json(selection_path, {"neural_grid_inner1600": inner_rows, "selected": inner_pick, "old_pick_frozen2000": OLD_PICK,
        "same_as_old": W == OLD_PICK, "reference_stats_6400_fit": {"classical": c_stats, "electra": e_stats},
        "frozen2000_read": False, "utc": datetime.now(timezone.utc).isoformat()})
    artifacts.append(selection_path)
print(f"Pick on inner 1600: w = {W} (old #18 pick on frozen 2000: {OLD_PICK})")
display(pd.DataFrame(inner_rows).set_index("neural_weight").round(6))

In [ ]:
# Frozen 2000 confirmation with saved 8000-fit scores. The pick is already saved.
with guarded("frozen_confirmation"):
    load = lambda key: pd.read_csv(ROOT / inputs[key], dtype={"id": "string"})
    c_ref, e_ref = load("17_unlabeled"), load("16_unlabeled")
    c8, e8 = ref_stats(c_ref.blend_score), ref_stats(e_ref.margin_original)
    e_val = load("16_validation").set_index("id").loc[validation.id]
    classical_val, neural_val = {}, {}
    for v in VIEWS:
        frame = load(f"17_validation_{v}")
        require(frame.id.tolist() == validation.id.tolist(), "Validation order differs")
        classical_val[v], neural_val[v] = standardize(frame.blend_score, c8), standardize(e_val[f"margin_{ELECTRA_VIEW[v]}"], e8)
    frozen_rows, _ = neural_grid(validation.label.to_numpy(), classical_val, neural_val)
    frozen_by_w = {r["neural_weight"]: r for r in frozen_rows}
    require(abs(frozen_by_w[OLD_PICK]["mean_auroc"] - dep18["metrics"]["neural_selected"]["mean_auroc"]) <= 1e-12, "Frozen grid differs from #18")

# Test rescore with the saved #18 10000-fit models.
with guarded("test_rescore"):
    t = time.perf_counter()
    with open(ROOT / full_files[0], "rb") as handle:
        full = cloudpickle.load(handle)
    electra10 = AutoModelForSequenceClassification.from_pretrained(ROOT / full_dir / "electra_10000", local_files_only=True).to(DEVICE)
    def test_scores(texts):
        texts = list(texts)
        c = standardize(classical_score(full["style"], full["old_soft"], texts), full["stats"]["classical"])
        e = standardize(electra_margins(electra10, texts), full["stats"]["electra"])
        return c, e
    c_test, e_test = test_scores(test.text)
    submitted = pd.read_csv(ROOT / full_files[3], dtype={"id": "string"}).set_index("id").label
    old_p = pd.Series(expit((1 - OLD_PICK) * c_test + OLD_PICK * e_test), index=test.id)
    repro_diff = float(np.max(np.abs(old_p.loc[submitted.index].to_numpy() - submitted.to_numpy())))
    require(repro_diff <= 1e-6, f"Saved #18 models do not reproduce the submitted file: {repro_diff}")
    new_submission = None
    if W != OLD_PICK:
        new_p = expit((1 - W) * c_test + W * e_test)
        order = np.random.default_rng(SEED).permutation(len(test))
        c_s, e_s = test_scores(test.text.iloc[order])
        recomputed = dict(zip(test.id.iloc[order], expit((1 - W) * c_s + W * e_s)))
        out_dir = ROOT / "submissions" / NAME / RUN_ID; out_dir.mkdir(parents=True, exist_ok=False)
        frame = pd.DataFrame({"id": sample.id}).merge(pd.DataFrame({"id": test.id, "label": new_p}), on="id", how="left", validate="one_to_one")
        frame.to_csv(out_dir / "submission.csv", index=False)
        new_submission = check_submission(out_dir / "submission.csv", sample.id.tolist(), recomputed)
        new_submission.update({"sha256": file_hash(out_dir / "submission.csv"), "rank_spearman_vs_submitted":
                               float(pd.Series(new_p, index=test.id).loc[submitted.index].rank().corr(submitted.rank()))})
        artifacts.append(out_dir / "submission.csv")
    timings["test_rescore_seconds"] = time.perf_counter() - t
    del electra10
timings["run_seconds"] = time.perf_counter() - START
print(f"Saved #18 models reproduce the submitted file: max diff {repro_diff:.3g}")
print("New submission:" if new_submission else "Pick unchanged, no new submission.", new_submission or "")

## Results

Inner 1600 are training rows, so this pick follows the rule. Frozen 2000 is confirmation only, after the pick, and was reused before. ELECTRA on MPS is not bitwise repeatable, so its inner AUROC can differ a little from #16. None of this is a Kaggle score.

In [ ]:
print("Grid on inner 1600 (6400-fit, training rows). This picks w:")
display(pd.DataFrame(inner_rows).set_index("neural_weight").round(6))
print(f"Picked w = {W}. #18 picked {OLD_PICK} on frozen 2000. Same: {W == OLD_PICK}")
print("Frozen 2000 grid (8000-fit, confirmation only, same numbers as #18):")
display(pd.DataFrame(frozen_rows).set_index("neural_weight").round(6))
print("Classical inner reproduction vs #17, max AUROC diff:", max(classical_repro.values()))
print(f"ELECTRA inner original AUROC {electra_inner_auroc['original']:.6f} vs #16 {saved16:.6f}")
display(pd.DataFrame(electra_history).round(6))
display(pd.DataFrame([{"stage": k, "seconds": v} for k, v in {**{f"classical_fit_{k}": s for k, s in classical_fit_seconds.items()}, **timings}.items()]).round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
for axis, rows, title in [(ax[0], inner_rows, "Inner 1600 (picks w)"), (ax[1], frozen_rows, "Frozen 2000 (confirmation)")]:
    grid = pd.DataFrame(rows)
    for v in VIEWS + ["mean_auroc"]:
        axis.plot(grid.neural_weight, grid[v], marker="o", label=v)
    axis.axvline(W, color="gray", ls="--", lw=1); axis.set(title=title, xlabel="ELECTRA weight w", ylabel="AUROC"); axis.legend(fontsize=8)
fig.tight_layout(); plot_path = ROOT / "results" / NAME / f"{RUN_ID}-grid.png"; fig.savefig(plot_path, dpi=130); artifacts.append(plot_path); plt.show()

## Takeaway

- Tried: TODO after run.
- Found: TODO after run.
- Next: TODO after run.

## Save

The safe record keeps both grids, the pick, reproduction checks, ELECTRA history, and timings. No model is saved; the 6400-fit models are diagnostics. A new submission is written only if the pick changes, under ignored `submissions/`. Nothing is uploaded.

In [ ]:
metrics = {"validation_auroc": None, "neural_grid_inner1600": inner_rows, "neural_selected": inner_pick, "old_pick_frozen2000": OLD_PICK,
    "same_as_old": W == OLD_PICK, "frozen2000_grid_confirmation": frozen_rows, "frozen2000_selected": frozen_by_w[W],
    "classical_inner_repro_max_diff": max(classical_repro.values()), "electra_inner_auroc": electra_inner_auroc,
    "electra_inner_original_16": saved16, "electra_history": electra_history, "old_soft_rule_active_6400": old6.rule_active_,
    "reference_stats_6400_fit": {"classical": c_stats, "electra": e_stats},
    "submitted_file_repro_max_diff": repro_diff, "new_submission": new_submission,
    "fit_seconds": classical_fit_seconds, "timings": timings,
    "kaggle": {"uploaded_by_notebook": False}}
heldout = {"rows": "frozen 2000, saved 8000-fit scores, read after the pick", "selection_record": str(selection_path.relative_to(ROOT)),
           "picked_blend": frozen_by_w[W], "note": "frozen 2000 reused in earlier issues, so optimistic"}
record_path = record_run(NAME, COMMAND, CONFIG, SEED, 6400, 1600, metrics, time.perf_counter() - START, artifacts=artifacts, heldout_confirmation=heldout)
print(f"Saved {record_path.relative_to(ROOT)}")